# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# ローカル環境(LAN の検証機)—— `KM_ENV=local`

**本番と同じ構成を、LAN の中の Ubuntu 機に立てる。** 設定を試す・控えから戻す練習をする・本番に当てる前に通す、のための場所。

**本番の構成をそのまま持ち込んではいけない。** 2026-09-17 に実際に踏んだ: Let's Encrypt の証明書が取れず(公開 DNS が無い)、
**nginx が証明書を読めずに落ちる → `restart: unless-stopped` で再起動を繰り返した。** MariaDB も `certs/` の証明書が無いと同じく落ちる。

そこで、**ファイルを複製せず、`.env` の切り替えで**立てる。検証機で `scripts/host-local.sh init` を 1 回流すと:

| 作るもの | 中身 |
|---|---|
| `.env` の 3 行 | `KM_ENV=local` / `KM_DOMAIN=<ローカルの名前>` / `COMPOSE_FILE=compose.yaml:compose.vps.yaml:compose.local.yaml` |
| `certs/` | 自作 CA(10 年)と、MariaDB・nginx の証明書(397 日。名前が変わるか残り 30 日で作り直す。古いものは `certs/Old/` へ) |
| `nginx/km/allow-admin-home.local.conf` | 管理系ポート(8281・3002・8025)に入れる LAN の範囲 |

`compose.local.yaml` が本番の 2 枚に重なり、**証明書(自作 CA)・メール(Mailpit)・コンテナの中の名前解決・CA の信頼**だけを差し替える。
read_only・網の分割・上限などは本番と同じまま(置き換えずに足すだけ。`check.php` が見張る)。管理画面の右下に **「ローカル環境」** と出る。

**本番では止まる。** `host-local.sh` は、本番の `.env`・Let's Encrypt の証明書があるホスト・公開のアドレスを指す名前のどれかなら何もしない。
`deploy-to-host.ps1` も、相手の `.env` の `KM_ENV` を表示し、本番に `KM_ENV=local` があれば止まる。

> 使い方は [00-start.ipynb](00-start.ipynb)。**下のセルの `KM_HOST` を検証機に書き換えてから 1 回実行する。**
> このノートの `%%host` と `%%ps` は、その `KM_HOST` へ繋ぐ(本番へは繋がない)。

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する |
| 🔴 | **検証機が変わる。** 実行前に `yes` の入力を求める(このノートでは本番ではなく検証機) |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する。**KM_HOST を検証機に書き換えてから**(%%host / %%ps がそこへ繋ぐ。既定のままだと本番へ繋ぐ)
import os, sys, pathlib, importlib
os.environ['KM_HOST'] = '192.168.217.128'   # ← 検証機の IP か名前
os.environ['KM_USER'] = 'km'      # ← §1 で kmops を作って渡したら 'kmops' に変える(本番と同じ形。12 §7-4 B)
os.environ['KM_KEY'] = str(pathlib.Path.home() / '.ssh' / 'test')   # ← 検証機に入る鍵(setdefault にしない。先に別のノートを読み込んでいると本番の鍵が残る)
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
importlib.reload(km_nb)   # 別のノートで本番向けに読み込んだあとでも、KM_HOST を読み直す
km_nb.load()

### 繋いだ先が検証機か

**この先のセルを流す前に必ず。** 本番のホストなら止まる。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
echo "繋いだ先: $(hostname) / $(hostname -I | awk '{print $1}')"
case " $(hostname -I) " in
  *" 163.43.218.158 "*) echo "★ ここは本番のホストです。最初のセルの KM_HOST を検証機に書き換えて、やり直してください"; exit 1 ;;
esac
if [ -f .env ]; then
  echo "KM_ENV=$(sed -n 's/^KM_ENV=//p' .env | tail -n 1)(local ならローカル環境。空なら init の前)"
else
  echo ".env はまだありません(§1 で作る)"
fi

## 1. 検証機を用意する

- **Ubuntu Server 24.04 LTS + Docker**。入れ方は [09-new-host](09-new-host.ipynb) の前半(利用者 `km`・SSH の鍵・docker グループ)と同じ。**本番と同じ形にするなら、最初の配備のあとに `kmops` を作って置き場を渡す**(下の §1-2。[12](12-hardening-2026-09-15.ipynb) §7-4 B)。**証明書と DNS の段は飛ばす**
- メモリは本番と同じ 2GB 以上
- ファイアウォールは **LAN からだけ** 80・443・3001・3002・6001・8025・8281 を通す
- 検証機の LAN の IP は固定にする(ルーターの DHCP の予約)。証明書と名前がその IP を指すため

置き場 `/opt/kosenmap` と、空の `.env`(`deploy-to-host.ps1` は `.env` が無いと送らない)を作る。`/opt` は root の持ち物なので sudo が要る。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:KM_KEY" "km@$env:KM_HOST" "sudo install -d -o km -g km -m 755 /opt/kosenmap && (umask 077 && touch /opt/kosenmap/.env) && ls -la /opt/kosenmap; read -r -p 'Enter で閉じる' _"

## 2. 名前を決める

**PC のブラウザと Android の両方から引ける、点を含む名前**が要る(Android のビルドは `localhost` や IP を受け付けない。端末には hosts も無い)。

| やり方 | 例 | 向き・不向き |
|---|---|---|
| **LAN のルーター / DNS に登録する(勧める)** | `km.test` → `192.168.1.50` | 外に頼らない。ルーターに「ローカル DNS」「固定 DNS エントリ」の設定があること |
| sslip.io を使う | `192-168-1-50.sslip.io` | 登録が要らない(公開の DNS が名前の中の IP を返す)。ルーターの **DNS rebinding 保護**で引けないことがある |
| PC の hosts だけ | `km.test` | ブラウザだけで足りるとき。**Android からは引けない** |

**本番の名前(`ito4.jp`・旧 `ito8795.com` とそれらのサブドメイン)は使わない。** 管理画面を別オリジンにして試すなら、管理用の名前(例 `admin.km.test`)も同じように引けるようにする。

## 3. コードを送る

`-Action deploy`(ファイルを置くだけ)。送る前に、相手の `.env` に `KM_ENV=local` が**まだ無い**ので「本番の構成として動きます」と黄色で出る —— §4 の前なので正しい。
初めて繋ぐ検証機は、指紋を確かめて `known_hosts` に登録する(`-AcceptHostKey`)。

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%ps --confirm "検証機へコードを置きます(本番には送りません。KM_HOST の先だけ)"
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
.\deploy-to-host.ps1 -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY -Action deploy -AcceptHostKey -Yes

## 4. ローカル環境に切り替える(`host-local.sh init`)

`DOMAIN` を §2 で決めた名前にする。管理画面を別オリジンでも試すなら `--admin-domain` を足す。
**何度流してもよい**(CA は使い回し、証明書は名前か期限が変わったときだけ作り直す)。検証機の IP が変わったら `--ip` を付けて流し直す。

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%host --confirm "検証機の .env をローカル環境に切り替え、自作 CA と証明書を作ります"
DOMAIN=km.test   # ← §2 で決めた名前
sh scripts/host-local.sh init --domain "$DOMAIN" </dev/null

## 5. 控えから秘密の値と設定を持っていく

使う控えは `D:\Backups\` の**いちばん新しい `ito4.jp-*`**(下のセルの例は 2026-09-18 のもの)。

1. この PC で控えを開く(`-Keep` で開いたままにする。**開いた平文は 24 時間で世代整理が消す**)
2. `env.txt`(本番の .env)と `km-config.tar.gz`(`config/*.local.php`)を検証機へ送る
3. 検証機で .env に足す(**`KM_ENV`・`KM_DOMAIN`・`COMPOSE_FILE`・`LOGTO_DB_*` は足さない**)・設定を置く・送った写しを消す → `init` をもう一度流して、本番の URL が残った行を空にする

**本番の Logto の資格情報(アプリの ID と秘密)も含めて写る。** 検証機は LAN から出さない。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
.\open-backup.ps1 -Path 'D:\Backups\ito4.jp-20260918-020229\km-backup-ubuntu-20260918-020229.tar.gz.cms' -Keep | Format-List

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%ps --confirm "開いた控えの env.txt と config を検証機へ送ります(秘密を含む。着いたらすぐ 600 にします)"
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
$opened = (Get-ChildItem 'D:\Backups\_opened' -Directory | Sort-Object Name -Descending | Select-Object -First 1).FullName
if (-not $opened) { throw '開いた控えがありません(上のセル)' }
. .\km-ssh.ps1
Initialize-KmSsh -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY
Invoke-KmSshStdin -Script 'umask 077; : > /opt/kosenmap/.env.from-backup; : > /opt/kosenmap/km-config.from-backup.tar.gz' -TimeoutSec 30 | Out-Null
Copy-KmTo -Path (Join-Path $opened 'env.txt') -Destination '/opt/kosenmap/.env.from-backup'
Copy-KmTo -Path (Join-Path $opened 'km-config.tar.gz') -Destination '/opt/kosenmap/km-config.from-backup.tar.gz'
Invoke-KmSshStdin -Script 'chmod 600 /opt/kosenmap/.env.from-backup /opt/kosenmap/km-config.from-backup.tar.gz' -TimeoutSec 30 | Out-Null
"送りました(開いた控え: $opened)"

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%host --confirm "控えの値を .env に足し、config/*.local.php を置き、送った写しを消し、init を流し直します"
set -eu
if [ ! -f .env.from-backup ] || [ ! -f km-config.from-backup.tar.gz ]; then echo "送った写しがありません(上のセル)"; exit 1; fi
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)"
added=0
tr -d '\r' < .env.from-backup | while IFS= read -r line; do
  case "$line" in ''|'#'*) continue ;; esac
  key="${line%%=*}"
  case "$key" in KM_ENV|KM_DOMAIN|KM_ADMIN_DOMAIN|COMPOSE_FILE|LOGTO_DB_USER|LOGTO_DB_PASSWORD) continue ;; esac
  if ! grep -q "^$key=" .env; then printf '%s\n' "$line" >> .env; fi
done
chmod 600 .env
echo ".env の行数: $(grep -c '=' .env)"
# config は www-data の持ち物になりうるので root を借りて置く(host-setup.sh と同じ手)。持ち主は次の host-setup --fix で揃う
mkdir -p src/config
docker run --rm -v "$PWD:/p" alpine sh -c '
  t=$(mktemp -d) && tar -xzf /p/km-config.from-backup.tar.gz -C "$t" &&
  find "$t" -name "*.local.php" -exec cp {} /p/src/config/ \; &&
  find "$t" -mindepth 1 -delete && rmdir "$t" && ls /p/src/config/*.local.php'
find . -maxdepth 1 \( -name .env.from-backup -o -name km-config.from-backup.tar.gz \) -delete
echo "送った写しを消しました"
sh scripts/host-local.sh init --domain "$(sed -n 's/^KM_DOMAIN=//p' .env | tail -n 1)" </dev/null | sed -n '/== 3/,/== 4/p'

## 6. DB を戻す

**DB だけを先に起動して戻す。** 全体を起動すると、Logto が空の DB に初期データを入れてしまい、`restore-data.ps1` が「空でない DB には入れない」で止まる。
`restore-data.ps1` は [03-backup](03-backup.ipynb) §8 と同じもの(**戻したあとに行数を突き合わせる**)。まず `-DryRun`(転送と検査だけ)。

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%host --confirm "検証機で MariaDB と Postgres だけを起動します(空の DB を作る)" --timeout 600
docker compose up -d mariadb postgres </dev/null
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps --timeout 1800
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
$opened = (Get-ChildItem 'D:\Backups\_opened' -Directory | Sort-Object Name -Descending | Select-Object -First 1).FullName
..\Old\restore-data.ps1 -BackupDir $opened -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY -DryRun

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%ps --confirm "検証機の DB に控えを戻します" --timeout 1800
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
$opened = (Get-ChildItem 'D:\Backups\_opened' -Directory | Sort-Object Name -Descending | Select-Object -First 1).FullName
..\Old\restore-data.ps1 -BackupDir $opened -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY

## 7. 起動する

1. `host-setup.ps1 -Fix` —— 持ち主と権限を揃え、**証明書が揃っているか**を見る(`★` が出たら up しない。再起動を繰り返す)
2. web と soketi をビルドして全体を up
3. `host-local.sh status` と、再起動の回数が 0 であることを見る

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%ps --confirm "検証機の持ち主と権限を揃えます"
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
# -Composer: src/vendor/ は配備で送らない(composer.lock から入れる)。無いとサインインが 500 になる
.\host-setup.ps1 -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY -Fix -Composer

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%host --confirm "検証機で web と soketi をビルドし、全体を起動します" --timeout 1800
docker compose build web soketi </dev/null
docker compose up -d </dev/null
# §5 で config/*.local.php を置き直すと、web のファイル単位の bind が外れていることがある
# (中から書けず、地図の錠・アクセスコードの保存だけが失敗する)。作り直して付け直す
docker compose up -d --force-recreate web </dev/null
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null

### 様子を見る

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 180
sh scripts/host-local.sh status </dev/null || true
echo
echo "== 再起動の回数(0 でないものは docker compose logs <サービス> で理由を見る)"
for c in $(docker compose ps -aq); do docker inspect -f '{{.Name}}  再起動 {{.RestartCount}}  {{.State.Status}} {{if .State.Health}}{{.State.Health.Status}}{{end}}' "$c"; done | sed 's#^/##' | sort

## 8. Logto をローカルの名前に向ける

控えから戻した Logto には、**本番の URL**(`https://ito8795.com/callback.php` など)が登録されている。**2026-09-17 の ito4.jp への統一より後の控えなら、下の 2 つのセルの `--from=` を `ito4.jp` にする。**ローカルの名前へ書き換える(**検証機の Logto だけ**が変わる)。

**確認コードのメール:** Logto の SMTP コネクタは本番の `mailserver` を向いている(ローカルでは起動しない)。
`https://<ローカルの名前>:3002`(Console)→ コネクタ → メール(SMTP)で、**ホスト `mailpit`・ポート `1025`・認証なし・TLS なし**にする。
届いたメールは `https://<ローカルの名前>:8025`(Mailpit)で読む。

**Logto の DB 利用者:** 戻した表は初期利用者の持ち物になっている。SUPERUSER でない利用者で試すなら [12](12-hardening-2026-09-15.ipynb) §3 のセルを、最初のセルで `KM_HOST` を検証機にしてから流す。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 180
docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito4.jp </dev/null

🔴 **検証機が変わる** —— 実行前に `yes` の入力を求めます(本番には繋ぎません。最初のセルの `KM_HOST` の先だけ)。

In [ ]:
%%host --confirm "検証機の Logto に登録された戻り先を、本番の名前からローカルの名前へ書き換えます" --timeout 180
docker compose exec -T -u www-data web php scripts/logto-domain.php --from=ito4.jp --apply </dev/null

## 9. この PC と Android から開く

### PC: CA を入れる

検証機の `certs/kosenmap-local-ca.crt` を取り寄せ、**この利用者の「信頼されたルート証明機関」**に入れる(Windows が確認の窓を出す)。
Edge と Chrome はここを使う。Firefox は別(設定 → 証明書 → 認証局 → インポート)。
**指紋を `host-local.sh status` の表示と見比べてから** 入れる。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%terminal
# 別の窓で開く。Windows が「この証明書をインストールしますか」と確認する窓を出すので、指紋が下と一致したら「はい」
if (-not $env:KM_HOST -or $env:KM_HOST -in @('ito4.jp', 'ito8795.com')) { throw '最初のセルで KM_HOST を検証機にしてください' }
. .\km-ssh.ps1
Initialize-KmSsh -HostName $env:KM_HOST -User $env:KM_USER -KeyPath $env:KM_KEY
$dest = Join-Path $env:USERPROFILE ".kosenmap\kosenmap-local-ca-$($env:KM_HOST).crt"
Copy-KmFrom -Path '/opt/kosenmap/certs/kosenmap-local-ca.crt' -Destination $dest
$cert = New-Object System.Security.Cryptography.X509Certificates.X509Certificate2 $dest
"主体: $($cert.Subject)"
"指紋(SHA256): $(($cert.GetCertHashString('SHA256') -split '(..)' | Where-Object { $_ }) -join ':')"
Import-Certificate -FilePath $dest -CertStoreLocation Cert:\CurrentUser\Root | Select-Object Subject, NotAfter

### PC: 名前を引けるようにする(ルーターの DNS に登録していないときだけ)

hosts に `<検証機の IP>  <ローカルの名前>` の 1 行を足す。管理者の権限が要る。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
Start-Process notepad -Verb RunAs -ArgumentList 'C:\Windows\System32\drivers\etc\hosts'

### Android(debug 版)

1. `kosenmap-local-ca.crt` を端末へ送る(USB・自分宛てのメールなど)
2. 設定 → セキュリティ → 暗号化と認証情報 → 証明書をインストール → **CA 証明書** → 送ったファイル
3. **debug 版**を、ローカルの名前でビルドして入れる(下のセル)。**release 版はユーザーの CA を信頼しない**(`network_security_config.xml`)
4. 端末の DNS でローカルの名前が引けること(§2)。引けなければ sslip.io の名前で init からやり直す

`kosenmap.apiResource` は戻した Logto の登録と同じにする(**名前であって接続先ではない**)。統一より前の控え(`D:\Backups\ito8795.com-…`)なら `-Pkosenmap.apiResource=https://ito8795.com/api` と、検証機の `.env` に `KM_API_RESOURCE=https://ito8795.com/api`。後の控えなら既定(`https://ito4.jp/api`)のまま。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps --timeout 1800
$domain = 'km.test'   # ← §2 で決めた名前(https:// もポートも付けない)
$staffOrg = 'vpguys770lr3'   # ← 検証機の Logto の教職員の組織 ID(docs/15。本番とは違う)
# ↓ 検証機の .env の KM_API_RESOURCE と同じ値。控えから戻した検証機は旧ドメインの名前のまま(2026-09-18 に実測)。
#   アプリの既定(https://ito4.jp/api)のまま作ると、サインインが invalid_target で失敗する
$apiResource = 'https://ito8795.com/api'
$jdk = Get-ChildItem 'C:\Program Files\Eclipse Adoptium' -Directory -Filter 'jdk-21*' -ErrorAction SilentlyContinue |
    Sort-Object Name -Descending | Select-Object -First 1
if (-not $jdk) { throw 'JDK 21 が見つかりません' }
$env:JAVA_HOME = $jdk.FullName
Set-Location (Join-Path $env:USERPROFILE 'Documents\Test')
.\gradlew.bat assembleVisitorDebug assembleAdminDebug --no-configuration-cache --console=plain "-Pkosenmap.domain=$domain" "-Pkosenmap.staffOrgId=$staffOrg" "-Pkosenmap.apiResource=$apiResource"
if ($LASTEXITCODE -ne 0) { exit $LASTEXITCODE }
Get-ChildItem app\build\outputs\apk -Recurse -Filter *debug*.apk | Select-Object FullName, Length, LastWriteTime

## 10. 確かめる

- 通しの確認は [12](12-hardening-2026-09-15.ipynb) §2-4 のセル(**最初のセルで `KM_HOST` を検証機に**)
- 画面の確認は 12 の §1 の一覧。加えて:
  - [ ] 管理画面の右下に **「ローカル環境」** と出る
  - [ ] 問い合わせを送ると **Mailpit(`:8025`)に届く**(外へは出ない)
  - [ ] Logto のサインインの確認コードが Mailpit に届く
  - [ ] debug 版のアプリでサインインし、地図を取れる

## 11. 困ったとき

| 症状 | 見るところ | 打つ手 |
|---|---|---|
| コンテナが再起動を繰り返す | §7 の「様子を見る」の再起動の回数 → `docker compose logs <サービス>` | nginx / mariadb なら証明書。`host-local.sh status` の ★。**`certs/` の中にディレクトリができていたら**、ファイルが無いまま up した跡 —— 退避して `init` |
| ブラウザが「この接続ではプライバシーが保護されません」 | 入れた CA の指紋と `status` の指紋 | CA を入れ直す。名前を変えたなら `init --domain <新しい名前>` と `nginx -s reload` |
| 管理画面が 503「認証の設定が未完了」・JWKS を取れない | web のログ | web の中から名前が引けない(compose.local.yaml の別名)か CA を信頼していない(`99-local-ca.ini`)。`docker compose up -d` し直す |
| サインインが `redirect_uri` の不一致 | §8 | `logto-domain.php --apply` をまだ流していない |
| 確認コードが届かない | Mailpit(`:8025`) | Logto の SMTP コネクタを `mailpit:1025` に(§8) |
| 8281・3002・8025 が 403 | `nginx/km/allow-admin-home.local.conf` | PC の IP が LAN の範囲外。`init --lan <範囲>` |
| 証明書の期限(397 日) | `host-local.sh status` | `init` を流し直す(残り 30 日を切っていれば作り直す)→ `docker compose exec reverse-proxy nginx -s reload` |
| 検証機の IP が変わった | `status` の SAN | `init --domain <名前> --ip <新しい IP>`、ルーターの DNS / hosts も直す |